# Cross-Lingual NLI with XLM-RoBERTa: End-to-End Pipeline

Final project for *Introduction to Computational Linguistics* — University of Vienna, Winter 2026.  
**Elisa Lopes de Souza**

This notebook walks through the full pipeline: data loading, four fine-tuning experiments, final test-set evaluation, embedding extraction for visualization, and qualitative error analysis. The heavy lifting lives in the `src/` package so the logic can also be run from the command line via the scripts in `scripts/`.

## 1. Setup

In [ ]:
# Colab only — skip if you installed via requirements.txt
# !pip install -q transformers datasets evaluate accelerate tensorboard scikit-learn

In [ ]:
# When running in Colab, clone the repo first so the src/ package is importable.
# !git clone https://github.com/elisalopes22/xlm-roberta-xnli-cross-lingual.git
# %cd xlm-roberta-xnli-cross-lingual

import os, sys

sys.path.insert(0, os.path.abspath(".."))

In [ ]:
from src.config import EXPERIMENTS, BEST_EXPERIMENT, LABELS, SEED
from src.data import (
    load_xnli_splits,
    build_mixed_train,
    build_mixed_validation,
    tokenize_splits,
)
from src.train import build_trainer, load_tokenizer, set_seed
from src.evaluate import evaluate_per_language, pretty_print_results
from src.error_analysis import collect_predictions, summarise
from src.embeddings import extract_layer_embeddings

set_seed(SEED)
OUTPUT_ROOT = "./results/checkpoints" 

## 2. Load XNLI and build mixed EN+TR splits

Validation is deliberately built by concatenating the EN and TR validation splits so the model is monitored on both languages at every epoch.

In [ ]:
tokenizer = load_tokenizer()
xnli_en, xnli_tr = load_xnli_splits()

val_multi = build_mixed_validation(xnli_en, xnli_tr)
print(f"Mixed validation size: {len(val_multi)}")

tokenized_val = tokenize_splits(
    {"en": xnli_en["validation"], "tr": xnli_tr["validation"], "mixed": val_multi},
    tokenizer,
)

## 3. Four fine-tuning experiments

Rather than duplicating 30+ lines of `TrainingArguments` four times, each experiment is a declarative `ExperimentConfig` in `src/config.py` and `build_trainer` turns it into a configured `transformers.Trainer`.

| Experiment | Train | Epochs | LR | Weight decay | Purpose |
|---|---|---|---|---|---|
| Exp 1 | 5k | 4 | 2e-5 | 0.01 | Baseline |
| Exp 2 | 5k | 4 | 2e-5 | 0.07 | More regularisation |
| Exp 3 | 5k | 6 | 1e-5 | 0.01 | Slower, longer |
| Exp 4 | 10k | 4 | 2e-5 | 0.01 | **More data (best)** |

In [ ]:
trainers = {}

for key, cfg in EXPERIMENTS.items():
    print(f"\n{'=' * 70}\nRunning {cfg.name}\n{'=' * 70}")
    print(f"  {cfg.notes}")

    train_ds = build_mixed_train(xnli_en, xnli_tr, cfg.train_size_per_lang)
    tokenized_train = tokenize_splits({"train": train_ds}, tokenizer)["train"]

    trainer = build_trainer(
        experiment=cfg,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val["mixed"],
        output_root=OUTPUT_ROOT,
        tokenizer=tokenizer,
    )
    trainer.train()

    results = evaluate_per_language(
        trainer,
        {"EN": tokenized_val["en"], "TR": tokenized_val["tr"], "mixed": tokenized_val["mixed"]},
    )
    pretty_print_results(results, title=f"{cfg.name} validation")
    trainers[key] = trainer

## 4. Final evaluation on the held-out XNLI test set

The test set is used exactly once, on the best model selected from validation (Experiment 4).

In [ ]:
tokenized_test = tokenize_splits(
    {"en": xnli_en["test"], "tr": xnli_tr["test"]},
    tokenizer,
)

best_trainer = trainers[BEST_EXPERIMENT]

test_results = evaluate_per_language(
    best_trainer,
    {"English (source)": tokenized_test["en"], "Turkish (target)": tokenized_test["tr"]},
)
pretty_print_results(test_results, title="Final test-set accuracy")

gap = test_results["English (source)"] - test_results["Turkish (target)"]
print(f"\nEN -> TR transfer gap: {gap:+.4f}")

## 5. Hidden-state embeddings for the TensorBoard Projector

Extract the start-token (`<s>`) representation at three layers (input embeddings (0), middle encoder (6), and final encoder (12)), so the learned semantic geometry can be inspected visually.

For RoBERTa-based models the start token lives at position `[0]`, which is why `src/embeddings.py` indexes positionally instead of searching for a `[CLS]` token ID.

In [ ]:
premises = val_multi["premise"][:500]
hypotheses = val_multi["hypothesis"][:500]
labels = val_multi["label"][:500]

extract_layer_embeddings(
    model=best_trainer.model,
    tokenizer=tokenizer,
    premises=premises,
    hypotheses=hypotheses,
    labels=labels,
    output_dir="./results/embeddings",
    layers_of_interest=[0, 6, 12],
    max_examples=500,
)

# The generated TSV files were uploaded to https://projector.tensorflow.org

## 6. Qualitative error analysis

The Turkish error patterns discussed in the report (§5) surface most clearly on the Turkish validation split. The analysis below prints the first handful of errors and successes for each split, the report walks through specific morphological cases in depth.

In [ ]:
for name, ds in [
    ("Validation (Turkish)", tokenize_splits({"x": xnli_tr["validation"]}, tokenizer)["x"]),
    ("Test (Turkish)", tokenized_test["tr"]),
    ("Test (English)", tokenized_test["en"]),
]:
    records = collect_predictions(best_trainer, ds)
    summarise(records, dataset_name=name, num_errors=5, num_successes=3)

## 7. Takeaways

- Training data size was the single biggest lever: 5k → 10k improved validation accuracy by roughly 6 points, more than any regularisation or learning-rate adjustment achieved.
- The EN→TR test gap of 6.15 pp is consistent with XLM-R's known behavior on morphologically distant mid-resource targets under a constrained training budget.
- Errors cluster around cases where lexical overlap is weak — often driven by Turkish agglutination fragmenting roots across subword tokens, or by pragmatic ambiguity in machine-translated premises.